# Eye Cleaning Pipeline Validation

Geoff originally developed a matlab routine (`processEyeData.m`) located in `/matlabCode` with the help of Gemini to clean the eye data of both the raw Neon recording as well as the output
gaze positions of the egocentric video mapper. Zach then used GPT 6.1 Astra light to initially translate this into Python. After the initial translation was in place, Zach went through the code to validate correctness, readability, and design. This notebook serves as a validation of the performance between MATLAB and Python to ensure both pipeline's return the same results given the same inut.

We choose a small subset of test subjects and activities to run throug hboth pipelines. Note that the MATLAB engine requires Signal Processing Toolbox and to access these recordings, the GKALabNAS must be mounted.

For path purposes, this notebook assumes you are running it from the `analyzeEyeTracking` directory


## 1. Paths and settings

Choose the recordings and set the MATLAB path here. The two tolerances allow for small differences in floating-point calculations. They do not allow timestamps or missing-value locations to differ.


In [2]:
%matplotlib inline

import os
import subprocess
import tempfile
from pathlib import Path
from unittest.mock import patch
import warnings

import matplotlib.pyplot as plt
from matplotlib.figure import Figure
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.io import loadmat

import pythonCode.cleanEyeTracking as python_pipeline

# Path to the MATLAB executeable as well as the RAW and PROCESSED recording directories
matlab_executable: Path = Path("/Applications/MATLAB_R2025b.app/bin/matlab")
raw_recording_root: Path = Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026")
mapped_recording_root: Path = Path("/Volumes/FLIC_processing/NEWscriptedIndoorOutdoorVideos2026")

# Use every combination of subject and activity listed here.
subject_ids: tuple[int, ...] = (2001, 2002, 2003)
activity_names: tuple[str, ...] = ("chat", "walkIndoor", "walkOutdoor")

# Allow a small fixed error near zero and an error proportional to the
# MATLAB value elsewhere: absolute_tolerance + relative_tolerance * abs(value).
absolute_tolerance: float = 1e-9
relative_tolerance: float = 1e-10


## 2. Find the input CSVs

Each recording needs `3d_eye_states.csv`, `blinks.csv`, and `alternative_camera_gaze.csv`. Stop if a recording is missing. 


In [ ]:
def get_directory_names(parent_directory: Path) -> list[str]:
    """Return the names of directories directly inside the given path."""
    directory_names: list[str] = []

    for entry_name in os.listdir(parent_directory):
        entry_path: Path = parent_directory / entry_name

        if entry_path.is_dir():
            directory_names.append(entry_name)

    return directory_names


# Construct a list of dictionaries, each containing the recording name "subect/activity", 
# and path to its raw and processed csv folders 
recordings: list[dict] = []

# Iterate through the chosen subject ids 
for subject_id in sorted(subject_ids):

    # Iterate through the chosen activities
    for activity_name in sorted(activity_names):

        # Construct the paths to the folders
        recording_name: str = f"FLIC_{subject_id}/{activity_name}"
        raw_neon_folder: Path = raw_recording_root / recording_name / "Neon"
        mapped_neon_folder: Path = mapped_recording_root / recording_name / "Neon"

        # Both exports have a recording folder named 1 below Neon.
        # NOTE: Test subjects (2001-2006) ONLY have a 1. It's possible other subjects 
        # have other recordings and the first is not the intended recording
        raw_csv_folder: Path = raw_neon_folder / "1"
        mapped_csv_folder: Path = mapped_neon_folder / "1"
        

        # NOTE: Subjects not in our test subject set 
        #       have an additional layer of directories below each numerical recording 
        #       this is because they have both the timeseries and video output and native 
        #       recording output from Pupil Cloud. We did not have Pupil Cloud Premium when 
        #       these were recorded, hence they were deleted for space without their 
        #       "native recording" folders downloaded and are unfortunately 
        #       non-recoverable. What exists is the timeseries + world recording, no pupil camera 
        #       recording
        if(subject_id not in range(2001, 2007)):
            warnings.warn(f"Subject ID not in test subjects. Default is first recording, check if this is the intended directory")

            # Go down one directory further and get the named subdirectories
            # to append to the paht
            raw_directory_names: list[str] = get_directory_names(raw_csv_folder)
            mapped_directory_names: list[str] = get_directory_names(mapped_csv_folder)

            raw_csv_folder = raw_csv_folder / raw_directory_names[0]
            mapped_csv_folder = mapped_csv_folder / mapped_directory_names[0]

        mapped_csv_folder = mapped_csv_folder / "egocentric_video_mapper_results"

        # Store the two input folders together so both implementations use the
        # same recording later.
        recordings.append(
            {
                "recording_name": recording_name,
                "raw_csv_folder": raw_csv_folder,
                "mapped_csv_folder": mapped_csv_folder,
            }
        )

# Require all requested subject/activity pairs before reporting agreement.
expected_recording_count: int = len(subject_ids) * len(activity_names)
assert recordings, "No recordings selected."
assert len(recordings) == expected_recording_count, "Recording selection is incomplete."

display(pd.DataFrame(recordings))


,recording_name,raw_csv_folder,mapped_csv_folder
0,FLIC_2001/chat,/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVide...,/Volumes/FLIC_processing/NEWscriptedIndoorOutd...
1,FLIC_2001/walkIndoor,/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVide...,/Volumes/FLIC_processing/NEWscriptedIndoorOutd...
2,FLIC_2001/walkOutdoor,/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVide...,/Volumes/FLIC_processing/NEWscriptedIndoorOutd...
3,FLIC_2002/chat,/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVide...,/Volumes/FLIC_processing/NEWscriptedIndoorOutd...
4,FLIC_2002/walkIndoor,/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVide...,/Volumes/FLIC_processing/NEWscriptedIndoorOutd...
5,FLIC_2002/walkOutdoor,/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVide...,/Volumes/FLIC_processing/NEWscriptedIndoorOutd...
6,FLIC_2003/chat,/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVide...,/Volumes/FLIC_processing/NEWscriptedIndoorOutd...
7,FLIC_2003/walkIndoor,/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVide...,/Volumes/FLIC_processing/NEWscriptedIndoorOutd...
8,FLIC_2003/walkOutdoor,/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVide...,/Volumes/FLIC_processing/NEWscriptedIndoorOutd...


## 3. Run Geoff's MATLAB code

The MATLAB script reads CSVs from its current folder. For each recording, make a temporary folder with links to the original CSVs and run the script there. Load its results into the notebook, then remove the temporary files.

MATLAB runs all selected recordings in one session. Its messages and errors appear below.


In [ ]:
def quote_matlab_string(value: str | Path) -> str:
    """Quote a path for MATLAB, escaping embedded apostrophes."""
    escaped_value: str = str(value).replace("'", "''")
    return f"'{escaped_value}'"


matlab_source_path: Path = Path("matlabCode/processEyeData.m").resolve()
matlab_timeout_s: int = 600
matlab_results: dict[str, dict] = {}

# The reference reads three CSV names from its current folder. Use temporary
# links to bring them together without changing the original recording folders.
# The temporary script and MAT files are removed after loading their arrays.
with tempfile.TemporaryDirectory(prefix="eye_cleaning_matlab_") as temporary_directory:
    temporary_folder: Path = Path(temporary_directory)
    matlab_commands: list[str] = [
        "set(groot, 'defaultFigureVisible', 'off');",
        f"reference_script = fileread({quote_matlab_string(matlab_source_path)});",
    ]

    for recording_index, recording in enumerate(recordings):
        reference_folder: Path = temporary_folder / str(recording_index)
        reference_folder.mkdir()

        for filename, source_folder in (
            ("3d_eye_states.csv", recording["raw_csv_folder"]),
            ("blinks.csv", recording["raw_csv_folder"]),
            ("alternative_camera_gaze.csv", recording["mapped_csv_folder"]),
        ):
            csv_link: Path = reference_folder / filename
            csv_link.symlink_to(source_folder / filename)

        # Evaluate the unchanged source in the folder containing its inputs.
        # Save only the arrays needed for the notebook comparisons.
        matlab_commands.extend([
            f"cd({quote_matlab_string(reference_folder)});",
            "eval(reference_script);",
            "save('reference.mat', 'cleaned_data_matrix', 'pupil_raw', "
            "'aperture_mean_raw', 'aperture_diff_raw', 'gaze_x_raw', 'gaze_y_raw', "
            "'is_blink_raw', '-v7');",
            "close all force;",
        ])

    matlab_runner_path: Path = temporary_folder / "run_reference.m"
    matlab_runner_path.write_text("\n".join(matlab_commands) + "\n")

    # Keep MATLAB's messages in the cell output instead of writing a log file.
    matlab_process = subprocess.run(
        [str(matlab_executable), "-batch", f"run({quote_matlab_string(matlab_runner_path)})"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        timeout=matlab_timeout_s,
    )
    print(matlab_process.stdout)
    matlab_process.check_returncode()

    # Load each result into notebook memory before the temporary folder is removed.
    for recording_index, recording in enumerate(recordings):
        reference_path: Path = temporary_folder / str(recording_index) / "reference.mat"
        matlab_results[recording["recording_name"]] = loadmat(reference_path)


## 4. Run Python and compare the plotted values

Run the Python cleaner on those same CSVs. Check the data in each panel of its comparison figure against MATLAB: raw measurements, cleaned measurements, and the blink trace. The figures are displayed below and are not saved to separate files.


In [ ]:
# Column names follow the order returned by the cleaner.
output_columns: tuple[str, ...] = (
    "time [s]",
    "pupil diameter [mm]",
    "eyelid aperture [mm]",
    "eyelid aperture difference [mm]",
    "gaze x [px]",
    "gaze y [px]",
)
time_column: int = output_columns.index("time [s]")
measurement_columns: tuple[str, ...] = output_columns[1:]

# These MATLAB variables follow the same measurement order as the five
# Python plot panels. The sixth panel contains the blink trace.
matlab_raw_variables: tuple[str, ...] = (
    "pupil_raw",
    "aperture_mean_raw",
    "aperture_diff_raw",
    "gaze_x_raw",
    "gaze_y_raw",
)

validation_results: dict[str, dict[str, np.ndarray]] = {}

for recording in recordings:
    recording_name: str = recording["recording_name"]

    # Use the MATLAB arrays already loaded into the notebook.
    matlab_reference: dict = matlab_results[recording_name]
    matlab_data: np.ndarray = matlab_reference["cleaned_data_matrix"]

    # Remember already-open figures so we only inspect and close the new one.
    existing_figure_numbers: set[int] = set(plt.get_fignums())

    try:
        # Build the comparison figure without saving a PDF or displaying it
        # automatically. Display it below after checking its plotted values.
        with plt.ioff(), patch.object(plt, "show"), patch.object(Figure, "savefig"):
            python_data, cleaning_metadata = python_pipeline.clean_eye_data(
                recording["raw_csv_folder"],
                recording["mapped_csv_folder"],
                verbose=True,
            )

        # The difference between figure sets identifies the cleaner's figure,
        # without depending on whichever figure happened to be active.
        new_figure_numbers: set[int] = set(plt.get_fignums()) - existing_figure_numbers
        assert len(new_figure_numbers) == 1, f"{recording_name}: expected one comparison figure."

        comparison_figure = plt.figure(next(iter(new_figure_numbers)))
        assert len(comparison_figure.axes) == 6, "Expected six comparison panels."

        for panel_index, column_name in enumerate(measurement_columns):
            output_column: int = output_columns.index(column_name)
            matlab_raw_name: str = matlab_raw_variables[panel_index]
            raw_line, cleaned_line = comparison_figure.axes[panel_index].lines

            # Both lines must use exactly the same timestamps as the MATLAB output.
            # Otherwise matching y-values could still refer to different moments.
            for plotted_line in (raw_line, cleaned_line):
                np.testing.assert_array_equal(
                    plotted_line.get_xdata(),
                    matlab_data[:, time_column],
                    err_msg=f"{recording_name}: plotted timeline for {column_name}",
                )

            # Compare raw measurements against MATLAB's raw plot variables. ravel()
            # turns MATLAB row or column vectors into one-dimensional arrays.
            np.testing.assert_allclose(
                raw_line.get_ydata(),
                matlab_reference[matlab_raw_name].ravel(),
                atol=absolute_tolerance,
                rtol=relative_tolerance,
                equal_nan=True,
                err_msg=f"{recording_name}: raw plot for {column_name}",
            )

            # Compare the cleaned line with its named column in MATLAB's final matrix.
            # equal_nan=True accepts missing values only when they occur in both arrays.
            np.testing.assert_allclose(
                cleaned_line.get_ydata(),
                matlab_data[:, output_column],
                atol=absolute_tolerance,
                rtol=relative_tolerance,
                equal_nan=True,
                err_msg=f"{recording_name}: cleaned plot for {column_name}",
            )

            assert raw_line.get_marker() == ".", "Raw plot should use point markers."
            assert cleaned_line.get_marker() == "None", "Cleaned plot should use a line."

        # The last panel shows the original blink intervals, before padding.
        # Check its timestamps and Boolean values exactly.
        blink_panel_index: int = len(measurement_columns)
        blink_line = comparison_figure.axes[blink_panel_index].lines[0]
        np.testing.assert_array_equal(blink_line.get_xdata(), matlab_data[:, time_column])
        np.testing.assert_array_equal(
            blink_line.get_ydata(), matlab_reference["is_blink_raw"].ravel()
        )

        display(Markdown(f"### {recording_name}"))
        display(comparison_figure)

    finally:
        # Close newly created figures even when a check raises an error.
        for figure_number in set(plt.get_fignums()) - existing_figure_numbers:
            plt.close(figure_number)

    # Keep both final arrays for the numerical table and difference plots.
    validation_results[recording_name] = {"matlab_data": matlab_data, "python_data": python_data}

    print(f"{recording_name}: MATLAB {matlab_data.shape}; Python {python_data.shape}")

print("PASS: standard plot data agree for every selected recording.")


## 5. Compare the returned arrays

Compare each output column, including values that do not appear in a plot because they are missing. The table shows the largest absolute difference, the root-mean-square difference, and how many samples exceed the tolerance.

The table is displayed before checking for failures so it can help explain a failed comparison. `finite_pairs` shows how many samples could be compared numerically; a zero error with zero finite pairs does not mean there were valid measurements.


In [ ]:
comparison_rows: list[dict] = []

for recording in recordings:
    recording_name: str = recording["recording_name"]
    matlab_data: np.ndarray = validation_results[recording_name]["matlab_data"]
    python_data: np.ndarray = validation_results[recording_name]["python_data"]

    # The same shape is needed to compare corresponding samples and channels.
    assert matlab_data.shape == python_data.shape, f"{recording_name}: output shapes differ."
    assert matlab_data.shape[1] == len(output_columns), "Unexpected output column count."

    # Check time separately with no tolerance: a shifted timeline would make
    # the remaining sample-by-sample comparisons misleading.
    np.testing.assert_array_equal(
        python_data[:, time_column],
        matlab_data[:, time_column],
        err_msg=f"{recording_name}: elapsed timestamps differ",
    )

    for column_index, column_name in enumerate(output_columns):
        matlab_values: np.ndarray = matlab_data[:, column_index]
        python_values: np.ndarray = python_data[:, column_index]

        # Only calculate numerical errors where both implementations have finite
        # values. NaNs and infinities are checked separately below.
        finite_pair_mask: np.ndarray = np.isfinite(matlab_values) & np.isfinite(python_values)
        absolute_errors: np.ndarray = np.abs(
            python_values[finite_pair_mask] - matlab_values[finite_pair_mask]
        )

        # Scale the relative allowance by the MATLAB value. The fixed allowance
        # still permits small rounding differences when that value is near zero.
        allowed_errors: np.ndarray = absolute_tolerance + relative_tolerance * np.abs(
            matlab_values[finite_pair_mask]
        )

        # A missing value in only one result is a failure even if all finite
        # values match. Comparing these masks catches those excluded samples.
        missing_mismatch_mask: np.ndarray = np.isnan(python_values) != np.isnan(matlab_values)

        # Compare positive and negative infinity separately so a sign change is
        # also counted. OR combines them into one mismatch flag per sample.
        positive_infinity_mismatch_mask: np.ndarray = np.isposinf(python_values) != np.isposinf(
            matlab_values
        )
        negative_infinity_mismatch_mask: np.ndarray = np.isneginf(python_values) != np.isneginf(
            matlab_values
        )
        infinity_mismatch_mask: np.ndarray = (
            positive_infinity_mismatch_mask | negative_infinity_mismatch_mask
        )

        # An empty set of finite pairs has no numerical errors to summarize.
        # Keep zero here and report the number of finite pairs alongside it.
        maximum_absolute_error: float = 0.0
        root_mean_square_error: float = 0.0

        if len(absolute_errors) > 0:
            maximum_absolute_error = float(np.max(absolute_errors))

            # Squaring keeps errors from cancelling and gives larger differences
            # more weight. The square root returns the original measurement units.
            root_mean_square_error = float(np.sqrt(np.mean(absolute_errors**2)))

        # Collect one table row per recording and output column.
        comparison_rows.append(
            {
                "recording": recording_name,
                "signal": column_name,
                "samples": len(matlab_values),
                "finite_pairs": int(np.sum(finite_pair_mask)),
                "nan_mismatches": int(np.sum(missing_mismatch_mask)),
                "infinity_mismatches": int(np.sum(infinity_mismatch_mask)),
                "max_absolute_error": maximum_absolute_error,
                "rms_error": root_mean_square_error,
                "samples_outside_tolerance": int(np.sum(absolute_errors > allowed_errors)),
            }
        )

# Display the full table before raising any failures, so it can be inspected.
comparison_table: pd.DataFrame = pd.DataFrame(comparison_rows)

# Use scientific notation to make small floating-point differences legible.
with pd.option_context("display.float_format", "{:.3e}".format):
    display(comparison_table)

# All three checks must pass: matching missing values, matching infinities,
# and finite differences within tolerance.
assert comparison_table["nan_mismatches"].sum() == 0, "Missing-value locations differ."
assert comparison_table["infinity_mismatches"].sum() == 0, "Infinity locations or signs differ."
assert (
    comparison_table["samples_outside_tolerance"].sum() == 0
), "Numerical outputs exceed tolerance."

print(f"PASS: all six output columns agree for {len(recordings)} recordings.")


## 6. Look at the differences

For each measurement, plot the full recording, a close-up, and Python minus MATLAB. The close-up goes to the largest difference above the absolute tolerance. If there is no such difference, it shows the middle of the available data.


In [ ]:
# Width of the close-up around the selected sample, in seconds.
zoom_duration_s: float = 2.0

for recording in recordings:
    recording_name: str = recording["recording_name"]
    matlab_data: np.ndarray = validation_results[recording_name]["matlab_data"]
    python_data: np.ndarray = validation_results[recording_name]["python_data"]
    relative_timestamps_s: np.ndarray = matlab_data[:, time_column]

    # Each measurement gets a row with a full plot, close-up, and difference plot.
    figure, axes = plt.subplots(
        len(measurement_columns),
        3,
        figsize=(18, 14),
        constrained_layout=True,
        squeeze=False,
    )
    figure.suptitle(f"{recording_name} — MATLAB vs Python", fontweight="bold")

    for row_index, column_name in enumerate(measurement_columns):
        column_index: int = output_columns.index(column_name)
        matlab_values: np.ndarray = matlab_data[:, column_index]
        python_values: np.ndarray = python_data[:, column_index]

        # Keep the sign here: positive means the Python result is above MATLAB.
        measurement_difference: np.ndarray = python_values - matlab_values

        # Exclude missing and infinite differences when choosing where to zoom.
        finite_difference_mask: np.ndarray = np.isfinite(measurement_difference)
        finite_sample_indices: np.ndarray = np.where(finite_difference_mask)[0]

        # Default to the recording midpoint if neither implementation has a
        # finite value. Otherwise focus on the largest error or a middle sample.
        focus_sample_index: int = len(relative_timestamps_s) // 2
        if len(finite_sample_indices) > 0:
            finite_absolute_errors: np.ndarray = np.abs(
                measurement_difference[finite_sample_indices]
            )
            if np.max(finite_absolute_errors) > absolute_tolerance:
                largest_error_index: int = int(np.argmax(finite_absolute_errors))

                # argmax gives a position within the filtered errors. Convert it back
                # to the corresponding row in the full recording.
                focus_sample_index = int(finite_sample_indices[largest_error_index])
            else:
                focus_sample_index = int(finite_sample_indices[len(finite_sample_indices) // 2])

        # Center the requested time span on the focus sample. For short recordings,
        # limit the span to the recording duration.
        half_zoom_duration_s: float = zoom_duration_s / 2
        if len(relative_timestamps_s) > 1:
            recording_duration_s: float = relative_timestamps_s[-1] - relative_timestamps_s[0]
            half_zoom_duration_s = min(half_zoom_duration_s, recording_duration_s / 2)

        # Draw the same measurements in both overlays; only their x-limits differ.
        full_axis, detail_axis, difference_axis = axes[row_index]
        for axis in (full_axis, detail_axis):
            axis.plot(
                relative_timestamps_s,
                matlab_values,
                "k.-",
                linewidth=0.6,
                markersize=2,
                label="MATLAB",
            )
            axis.plot(relative_timestamps_s, python_values, "r--", linewidth=0.7, label="Python")
            axis.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize="small")

        focus_timestamp_s: float = float(relative_timestamps_s[focus_sample_index])
        full_axis.set_title(f"{column_name} — full recording")
        detail_axis.set_title("Detail")
        detail_axis.set_xlim(
            focus_timestamp_s - half_zoom_duration_s, focus_timestamp_s + half_zoom_duration_s
        )

        # Plot the signed difference separately because small errors may be
        # impossible to see when the two measurement lines overlap.
        difference_axis.plot(
            relative_timestamps_s, measurement_difference, color="tab:blue", linewidth=0.6
        )
        difference_axis.set_title("Python − MATLAB")

        # The first three measurements are physiology in mm; gaze is in pixels.
        measurement_units: str = "px" if column_name in ("gaze x [px]", "gaze y [px]") else "mm"
        for axis in axes[row_index]:
            axis.set_xlabel("Elapsed time [s]")
            axis.set_ylabel(measurement_units)
            axis.grid(alpha=0.25)

    display(figure)

    plt.close(figure)
